# 🌲 Module 10: Advanced Segmentation Architectures

Welcome to **Module 10** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Beyond Baseline U-Net**: Understand the limitations of standard skip connections on complex canopy boundaries.
2. **Attention Gates (Attention U-Net)**: Implement dynamic feature gating to amplify forest regions and suppress background artifacts.
3. **Atrous Spatial Pyramid Pooling (DeepLabV3-Lite)**: Use dilated convolutions to capture multi-scale context without resolution loss.
4. **Nested Dense Skips (U-Net++)**: Bridge semantic gaps between encoder and decoder sub-networks.
5. **Benchmarking & Latency**: Systematically compare parameters, CPU latency, IoU, and Dice score.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_09_unet_segmentation.unet_model import ForestUNet
from modules.module_09_unet_segmentation.segmentation_dataset import create_segmentation_dataloaders
from modules.module_10_advanced_segmentation.advanced_models import AttentionUNet, DeepLabV3Lite, NestedUNetLite
from modules.module_10_advanced_segmentation.model_comparator import compare_segmentation_models, count_parameters

%matplotlib inline
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Module 10 environment ready on device: [{device}]")

## 1. Load Multispectral Satellite DataLoaders

We load 5-band (Blue, Green, Red, NIR, SWIR) satellite scenes and paired binary forest masks.

In [ ]:
train_loader, val_loader, test_loader = create_segmentation_dataloaders(
    dataset_root="../dataset",
    tile_size=128,
    batch_size=8,
    max_train_tiles=24,
    max_val_tiles=8
)

print(f"Train tiles: {len(train_loader.dataset)} | Val tiles: {len(val_loader.dataset)} | Test tiles: {len(test_loader.dataset)}")

## 2. Inspect Architectures & Trainable Parameters

Let's compare the parameter footprints of the 4 models.

In [ ]:
models_to_test = {
    "Standard U-Net": ForestUNet(in_channels=5, out_channels=1, base_features=16),
    "Attention U-Net": AttentionUNet(in_channels=5, out_channels=1, base_features=16),
    "DeepLabV3-Lite": DeepLabV3Lite(in_channels=5, out_channels=1, base_features=16),
    "Nested U-Net++": NestedUNetLite(in_channels=5, out_channels=1, base_features=16),
}

for name, m in models_to_test.items():
    print(f"{name:<18}: {count_parameters(m):>9,} parameters")

## 3. Train & Benchmark All Models

We train each architecture under identical conditions (BCE + Dice Compound Loss, Adam optimizer, lr=1e-3).

In [ ]:
leaderboard_df, trained_models = compare_segmentation_models(
    models_dict=models_to_test,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    epochs=4,
    lr=1e-3,
    device=device
)

leaderboard_df

## 4. Visualizing Performance Trade-offs

Let's visualize the Mean IoU, Dice Score, CPU Latency, and Parameter efficiency.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

archs = leaderboard_df["Architecture"].tolist()
ious = [v * 100 for v in leaderboard_df["Mean IoU"]]
dices = [v * 100 for v in leaderboard_df["Dice Score"]]
latencies = leaderboard_df["Latency (ms)"].tolist()
params_k = [int(p.replace(",", "")) / 1000.0 for p in leaderboard_df["Parameters"]]

# 1. IoU & Dice
x = np.arange(len(archs))
width = 0.35
ax1.bar(x - width/2, ious, width, label="Mean IoU (%)", color="#2e7d32")
ax1.bar(x + width/2, dices, width, label="Dice (%)", color="#0288d1")
ax1.set_title("Mean IoU & Dice Score", fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(archs, rotation=15, ha="right")
ax1.set_ylim(0, 110)
ax1.legend()
ax1.grid(axis="y", linestyle="--", alpha=0.5)

# 2. Latency
ax2.bar(archs, latencies, color="#ff8f00", width=0.5)
ax2.set_title("Latency per Tile (ms)", fontweight="bold")
ax2.set_xticklabels(archs, rotation=15, ha="right")
ax2.grid(axis="y", linestyle="--", alpha=0.5)

# 3. Efficiency Scatter
ax3.scatter(params_k, ious, s=200, color="#8e24aa", edgecolors="black")
for i, arch in enumerate(archs):
    ax3.annotate(f" {arch}", (params_k[i], ious[i]), fontsize=9, fontweight="bold")
ax3.set_xlabel("Parameters (Thousands)", fontweight="bold")
ax3.set_ylabel("Mean IoU (%)", fontweight="bold")
ax3.set_title("Parameter Footprint vs IoU", fontweight="bold")
ax3.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Visualizing Attention Gate Coefficient Maps

Attention U-Net outputs attention maps $\alpha$ at each skip level, dynamically highlighting target canopies.

In [ ]:
def make_rgb(tensor_5ch):
    arr = tensor_5ch.cpu().numpy()
    def strt(b):
        p2, p98 = np.percentile(b, (2, 98))
        return np.clip((b - p2) / (p98 - p2 + 1e-8), 0.0, 1.0) if p98 > p2 else np.clip(b, 0.0, 1.0)
    return np.stack([strt(arr[2]), strt(arr[1]), strt(arr[0])], axis=-1)

att_model = trained_models["Attention U-Net"]
att_model.eval()

sample_x, sample_y = next(iter(test_loader))
with torch.no_grad():
    logits, att_maps = att_model.forward_with_attention_maps(sample_x[0:1].to(device))
    prob_map = torch.sigmoid(logits).cpu().numpy()[0, 0]

fig, axes = plt.subplots(1, 6, figsize=(20, 3.8))
axes[0].imshow(make_rgb(sample_x[0]))
axes[0].set_title("Input RGB")
axes[0].axis("off")

axes[1].imshow(sample_y[0, 0].numpy(), cmap="Greens")
axes[1].set_title("Ground Truth")
axes[1].axis("off")

for idx, (lvl, m_tensor) in enumerate(att_maps.items(), start=2):
    m_resized = F.interpolate(m_tensor, size=(128, 128), mode="bilinear", align_corners=False).cpu().numpy()[0, 0]
    axes[idx].imshow(m_resized, cmap="magma", vmin=0, vmax=1)
    axes[idx].set_title(f"Attention {lvl}")
    axes[idx].axis("off")

axes[5].imshow(prob_map, cmap="viridis", vmin=0, vmax=1)
axes[5].set_title("Predicted Prob")
axes[5].axis("off")

plt.tight_layout()
plt.show()